In [0]:
employee_df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True, sep="|", quote="'")
employee_df.display()

#Working with Array Types


##Create an array type column

In [0]:
from pyspark.sql.functions import split, col

result_df = (
    employee_df.withColumn("skills", split(col("col_skills"), ","))
    .withColumn(
        "current_expected_salary", split(col("col_current_expected_salary"), ",")
    )
    .drop("col_skills", "col_current_expected_salary")
)

result_df.display()
result_df.printSchema()

##Convert array of string to array of int

In [0]:
from pyspark.sql.functions import split, col

result_df = (
    employee_df.withColumn("skills", split(col("col_skills"), ","))
    .withColumn(
        "current_expected_salary", split(col("col_current_expected_salary"), ",").cast("array<int>")
    )
    .drop("col_skills", "col_current_expected_salary")
)

result_df.display()
result_df.printSchema()

##Accessing array elements

In [0]:
result_df.select(col("current_expected_salary"), col("current_expected_salary")[0].alias("actual_salary"), col("current_expected_salary").getItem(1).alias("expected_salary")).display()

In [0]:
result_df.withColumn(
    "actual_salary", col("current_expected_salary").getItem(0)
).withColumn("expected_salary", col("current_expected_salary").getItem(1)).display()

In [0]:
#get list of employee name whose expected salary is less than actual salary

result_df.filter(col("current_expected_salary").getItem(1) < col("current_expected_salary").getItem(0)).select("name").display()




##Array Functions

In [0]:
from pyspark.sql.functions import size, array_distinct, array_contains

result_df.select(
    col("name"),
    col("skills"),
    size(col("skills")),
    array_distinct(col("skills")),
    array_contains(col("skills"), "PySpark"),
).display()

In [0]:
#Select name of those for which skills contains PySpark
from pyspark.sql.functions import array_contains
result_df.filter(array_contains(col("skills"), "PySpark")==True).select(col("name"), col("skills")).display()

In [0]:
#select name of those for which skills contains PySpark and Hadoop and size of skills is greater than 3
from pyspark.sql.functions import size, array_contains

result_df.filter(
    (array_contains(col("skills"), "PySpark") == True)
    & (array_contains(col("skills"), "Hadoop") == True)
    & (size(col("skills")) > 3)
).select(col("name"), col("skills")).display()

#Struct Type


In [0]:
df = spark.read.json(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/product_Information_001.json",
    multiLine=True,
)
df.printSchema()
df.show()

In [0]:
df.select(col("details.features")).display()

In [0]:
df.select(col("details.screen.resolution")).display()

In [0]:
df.filter(array_contains(col("details.features"), "Heart Rate Monitor")).select(col("name")).display()

In [0]:
#explode creates a new row for each element in the array
from pyspark.sql.functions import explode
df.select(explode(col("details.features")).alias("features")).groupBy("features").count().display()

In [0]:
from pyspark.sql.functions import split
employee_df.withColumn("skills", split(col("col_skills"), ",")).select(explode(col("skills")).alias("top_skills")).groupby("top_skills").count().orderBy("count", ascending=False).limit(3).display()